In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import os
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display

In [17]:
file_path = os.path.join("..", "data", "processed", "feature_select.csv")

# Read CSV with index_col=0 to avoid unnamed column issues
data = pd.read_csv(file_path, index_col=0)
data

,Enrollment_ID,Study_Prog_Exam_Completed,Study_Prog_Group_Size,Student_Enrollment_Gap,Exit_Status,Study_Prog_Name_Accountancy,Study_Prog_Name_Allied_Medical_Care,Study_Prog_Name_Arts_Therapies,Study_Prog_Name_Bachelor_of_Law,Study_Prog_Name_Biology_Medical_Laboratory_Research,Study_Prog_Name_Built_Environment,Study_Prog_Name_Business_Administration,Study_Prog_Name_Cesar_Kinetics_Therapy,Study_Prog_Name_Chemical_Engineering,Study_Prog_Name_Chemistry,Study_Prog_Name_Communication,Study_Prog_Name_Communication_and_Multimedia_Design,Study_Prog_Name_Creative_Business,Study_Prog_Name_Dental_Hygiene,Study_Prog_Name_Educational_Theory,Study_Prog_Name_Electrical_and_Electronic_Engineering,Study_Prog_Name_Entrepreneurship_Retail_Management,Study_Prog_Name_Facility_Management,Study_Prog_Name_Finance_Control,Study_Prog_Name_Finance_Tax_and_Advice,Study_Prog_Name_Health_Care_Management,Study_Prog_Name_Human_Resource_Management,Study_Prog_Name_Industrial_Engineering_Management,Study_Prog_Name_Information_Communication_Technology,Study_Prog_Name_International_Business_Utrecht,Study_Prog_Name_Interpreter,Study_Prog_Name_Journalism,Study_Prog_Name_Logistics_Management,Study_Prog_Name_Marketing_Management,Study_Prog_Name_Mechanical_Engineering,Study_Prog_Name_Nursing,Study_Prog_Name_Optometry,Study_Prog_Name_Orthoptics,Study_Prog_Name_Pharmaceutical_Business_Administration,Study_Prog_Name_Physiotherapy,Study_Prog_Name_Primary_Education,Study_Prog_Name_Safety_and_Security_Management,Study_Prog_Name_Skin_Therapy,Study_Prog_Name_Social_Legal_Services,Study_Prog_Name_Social_Work,Study_Prog_Name_Speech_and_Language_Therapy,Study_Prog_Name_Teacher_Education,Prior_Edu_Type_BUITENL_SL,Prior_Edu_Type_EDUCATIE,Prior_Edu_Type_HAVO,Prior_Edu_Type_HBO,Prior_Edu_Type_MBO,Prior_Edu_Type_OVERIG,Prior_Edu_Type_Others,Prior_Edu_Type_VWO,Prior_Edu_School_Location_'S-GRAVENHAGE,Prior_Edu_School_Location_'S-HERTOGENBOSCH,Prior_Edu_School_Location_0000_ABROAD,Prior_Edu_School_Location_ALKMAAR,Prior_Edu_School_Location_ALMELO,Prior_Edu_School_Location_ALMERE,Prior_Edu_School_Location_ALPHEN_AAN_DEN_RIJN,Prior_Edu_School_Location_AMERSFOORT,Prior_Edu_School_Location_AMSTELVEEN,Prior_Edu_School_Location_AMSTERDAM,Prior_Edu_School_Location_APELDOORN,Prior_Edu_School_Location_ARNHEM,Prior_Edu_School_Location_ASSEN,Prior_Edu_School_Location_BAARN,Prior_Edu_School_Location_BARNEVELD,Prior_Edu_School_Location_BERGEN_NH,Prior_Edu_School_Location_BILTHOVEN,Prior_Edu_School_Location_BREDA,Prior_Edu_School_Location_BREUKELEN,Prior_Edu_School_Location_BUSSUM,Prior_Edu_School_Location_CAPELLE_AAN_DEN_IJSSEL,Prior_Edu_School_Location_CASTRICUM,Prior_Edu_School_Location_CULEMBORG,Prior_Edu_School_Location_DELFT,Prior_Edu_School_Location_DEVENTER,Prior_Edu_School_Location_DOETINCHEM,Prior_Edu_School_Location_DOORN,Prior_Edu_School_Location_DOORWERTH,Prior_Edu_School_Location_DORDRECHT,Prior_Edu_School_Location_EDE,Prior_Edu_School_Location_EINDHOVEN,Prior_Edu_School_Location_ELBURG,Prior_Edu_School_Location_ENSCHEDE,Prior_Edu_School_Location_ERMELO,Prior_Edu_School_Location_ETTEN-LEUR,Prior_Edu_School_Location_GORINCHEM,Prior_Edu_School_Location_GOUDA,Prior_Edu_School_Location_GRONINGEN,Prior_Edu_School_Location_HAARLEM,Prior_Edu_School_Location_HARDERWIJK,Prior_Edu_School_Location_HELMOND,Prior_Edu_School_Location_HENGELO_OV,Prior_Edu_School_Location_HILVERSUM,Prior_Edu_School_Location_HOOFDDORP,Prior_Edu_School_Location_HOORN_NH,Prior_Edu_School_Location_HOUTEN,Prior_Edu_School_Location_HUIZEN,Prior_Edu_School_Location_IJSSELSTEIN,Prior_Edu_School_Location_LAREN_NH,Prior_Edu_School_Location_LEERDAM,Prior_Edu_School_Location_LEEUWARDEN,Prior_Edu_School_Location_LEIDEN,Prior_Edu_School_Location_LELYSTAD,Prior_Edu_School_Location_MAARSSEN,Prior_Edu_School_Location_MAASTRICHT,Prior_Edu_School_Location_MIJDRECHT,Prior_Edu_School_Location_NIEUWEGEIN,Prior_Edu_School_Location_NIJKERK,Prior_Edu_School_Location_NIJMEGEN,Prior_Edu_School_Location_ONBEKEND,Prior_Edu_School_Location_Others,Pr

In [18]:
print(data['Exit_Status'].value_counts())

Exit_Status
leaving_unsuccessfully    43154
switching_internally       3489
leaving_successfully       1748
Name: count, dtype: int64


In [19]:
# Define a mapping for Exit_Status
exit_status_mapping = {
    'leaving_successfully': 0,
    'switching_internally': 1,
    'leaving_unsuccessfully': 2
}

# Apply the mapping to replace categorical values with numbers
data['Exit_Status'] = data['Exit_Status'].map(exit_status_mapping)

# Show the first few rows to verify
print(data['Exit_Status'].value_counts())


Exit_Status
2    43154
1     3489
0     1748
Name: count, dtype: int64


Print only the pairs of columns with correlation above 0.7

In [20]:
# Convert Exit_Status to numeric values automatically (if not already done)
data['Exit_Status'] = data['Exit_Status'].astype('category').cat.codes  # This will assign 0,1,2 automatically

# Drop Enrollment_ID (since it's just an identifier)
data_numeric = data.drop(columns=['Enrollment_ID'])

# Compute correlation matrix
corr_matrix = data_numeric.corr()

# Find pairs with correlation above 0.7 (excluding self-correlation)
high_corr_pairs = []
threshold = 0.7

for col1 in corr_matrix.columns:
    for col2 in corr_matrix.columns:
        if col1 != col2 and corr_matrix.loc[col1, col2] > threshold:
            high_corr_pairs.append((col1, col2, corr_matrix.loc[col1, col2]))

# Remove duplicate pairs (since correlation is symmetric)
unique_high_corr_pairs = set(tuple(sorted(pair[:2])) + (pair[2],) for pair in high_corr_pairs)

# Print highly correlated column pairs
print("Highly Correlated Column Pairs (Above 70%):")
for col1, col2, corr in unique_high_corr_pairs:
    print(f"{col1} - {col2}: {corr:.2f}")


Highly Correlated Column Pairs (Above 70%):
Prior_Edu_Postcode_3863_HV - Prior_Edu_School_Location_NIJKERK: 1.00
Prior_Edu_Postcode_8233_GA - Prior_Edu_School_Location_LELYSTAD: 1.00
Prior_Edu_Postcode_1507_EK - Prior_Edu_School_Location_ZAANDAM: 0.72
Prior_Edu_Postcode_3771_KA - Prior_Edu_School_Location_BARNEVELD: 1.00
Prior_Edu_Postcode_7001_EA - Prior_Edu_School_Location_DOETINCHEM: 0.74
Prior_Edu_Postcode_5616_HR - Prior_Edu_School_Location_EINDHOVEN: 0.75
Prior_Edu_Postcode_3452_MD - Prior_Edu_School_Location_VLEUTEN: 1.00
Prior_Edu_Postcode_7231_NN - Prior_Edu_School_Location_WARNSVELD: 1.00
Prior_Edu_Postcode_4205_NK - Prior_Edu_School_Location_GORINCHEM: 0.76
Prior_Edu_Postcode_1251_GB - Prior_Edu_School_Location_LAREN_NH: 1.00
Prior_Edu_Postcode_3723_BV - Prior_Edu_School_Location_BILTHOVEN: 0.71
Prior_Edu_Postcode_4001_DN - Prior_Edu_School_Location_TIEL: 0.80
Prior_Edu_Postcode_0000_ABROAD - Prior_Edu_School_Location_0000_ABROAD: 0.96
Prior_Edu_Postcode_6042_JZ - Prior_Edu_

Iteratively remove one of the columns from the highly correlated pairs until all remaining columns have a correlation of less than 0.7

In [22]:
# Drop Enrollment_ID (since it's just an identifier)
data_numeric = data.drop(columns=['Enrollment_ID'])

# Compute correlation matrix
corr_matrix = data_numeric.corr()

# Threshold for high correlation
threshold = 0.7

# Identify pairs of highly correlated columns
high_corr_pairs = set()
for col1 in corr_matrix.columns:
    for col2 in corr_matrix.columns:
        if col1 != col2 and corr_matrix.loc[col1, col2] > threshold:
            high_corr_pairs.add((col1, col2))

# Identify columns to remove to reduce correlation
columns_to_remove = set()
for col1, col2 in high_corr_pairs:
    if col1 not in columns_to_remove and col2 not in columns_to_remove:
        columns_to_remove.add(col2)  # Remove one of the highly correlated columns

# Keep only uncorrelated columns (less than 0.7 correlation)
final_columns = [col for col in data_numeric.columns if col not in columns_to_remove]
data_final = data_numeric[final_columns]

# Show the remaining columns and the first few rows
print("Remaining Columns After Removing High Correlation:")
print(data_final.columns)
print("\nData Preview:")
data_final


Remaining Columns After Removing High Correlation:
Index(['Study_Prog_Exam_Completed', 'Study_Prog_Group_Size',
       'Student_Enrollment_Gap', 'Exit_Status', 'Study_Prog_Name_Accountancy',
       'Study_Prog_Name_Allied_Medical_Care', 'Study_Prog_Name_Arts_Therapies',
       'Study_Prog_Name_Bachelor_of_Law',
       'Study_Prog_Name_Biology_Medical_Laboratory_Research',
       'Study_Prog_Name_Built_Environment',
       ...
       'Prior_Edu_Postcode_7414_AR', 'Prior_Edu_Postcode_8017_CA',
       'Prior_Edu_Postcode_8024_AH', 'Prior_Edu_Postcode_8031_AA',
       'Prior_Edu_Postcode_8081_VV', 'Prior_Edu_Postcode_8233_GA',
       'Prior_Edu_Postcode_8932_NX', 'Prior_Edu_Postcode_9723_ZS',
       'Prior_Edu_Postcode_Others', 'Prior_Edu_Country_NL'],
      dtype='object', length=246)

Data Preview:


,Study_Prog_Exam_Completed,Study_Prog_Group_Size,Student_Enrollment_Gap,Exit_Status,Study_Prog_Name_Accountancy,Study_Prog_Name_Allied_Medical_Care,Study_Prog_Name_Arts_Therapies,Study_Prog_Name_Bachelor_of_Law,Study_Prog_Name_Biology_Medical_Laboratory_Research,Study_Prog_Name_Built_Environment,Study_Prog_Name_Business_Administration,Study_Prog_Name_Cesar_Kinetics_Therapy,Study_Prog_Name_Chemical_Engineering,Study_Prog_Name_Chemistry,Study_Prog_Name_Communication,Study_Prog_Name_Communication_and_Multimedia_Design,Study_Prog_Name_Creative_Business,Study_Prog_Name_Dental_Hygiene,Study_Prog_Name_Educational_Theory,Study_Prog_Name_Electrical_and_Electronic_Engineering,Study_Prog_Name_Entrepreneurship_Retail_Management,Study_Prog_Name_Facility_Management,Study_Prog_Name_Finance_Control,Study_Prog_Name_Finance_Tax_and_Advice,Study_Prog_Name_Health_Care_Management,Study_Prog_Name_Human_Resource_Management,Study_Prog_Name_Industrial_Engineering_Management,Study_Prog_Name_Information_Communication_Technology,Study_Prog_Name_International_Business_Utrecht,Study_Prog_Name_Interpreter,Study_Prog_Name_Journalism,Study_Prog_Name_Logistics_Management,Study_Prog_Name_Marketing_Management,Study_Prog_Name_Mechanical_Engineering,Study_Prog_Name_Nursing,Study_Prog_Name_Optometry,Study_Prog_Name_Orthoptics,Study_Prog_Name_Pharmaceutical_Business_Administration,Study_Prog_Name_Physiotherapy,Study_Prog_Name_Primary_Education,Study_Prog_Name_Safety_and_Security_Management,Study_Prog_Name_Skin_Therapy,Study_Prog_Name_Social_Legal_Services,Study_Prog_Name_Social_Work,Study_Prog_Name_Speech_and_Language_Therapy,Study_Prog_Name_Teacher_Education,Prior_Edu_Type_BUITENL_SL,Prior_Edu_Type_EDUCATIE,Prior_Edu_Type_HAVO,Prior_Edu_Type_HBO,Prior_Edu_Type_MBO,Prior_Edu_Type_OVERIG,Prior_Edu_Type_Others,Prior_Edu_Type_VWO,Prior_Edu_School_Location_'S-GRAVENHAGE,Prior_Edu_School_Location_ALMELO,Prior_Edu_School_Location_ALMERE,Prior_Edu_School_Location_ALPHEN_AAN_DEN_RIJN,Prior_Edu_School_Location_AMERSFOORT,Prior_Edu_School_Location_AMSTELVEEN,Prior_Edu_School_Location_APELDOORN,Prior_Edu_School_Location_ASSEN,Prior_Edu_School_Location_BAARN,Prior_Edu_School_Location_BARNEVELD,Prior_Edu_School_Location_BERGEN_NH,Prior_Edu_School_Location_BREDA,Prior_Edu_School_Location_BUSSUM,Prior_Edu_School_Location_CAPELLE_AAN_DEN_IJSSEL,Prior_Edu_School_Location_DELFT,Prior_Edu_School_Location_EDE,Prior_Edu_School_Location_ENSCHEDE,Prior_Edu_School_Location_ERMELO,Prior_Edu_School_Location_ETTEN-LEUR,Prior_Edu_School_Location_GORINCHEM,Prior_Edu_School_Location_GOUDA,Prior_Edu_School_Location_GRONINGEN,Prior_Edu_School_Location_HAARLEM,Prior_Edu_School_Location_HELMOND,Prior_Edu_School_Location_HENGELO_OV,Prior_Edu_School_Location_HILVERSUM,Prior_Edu_School_Location_HOOFDDORP,Prior_Edu_School_Location_HOORN_NH,Prior_Edu_School_Location_HOUTEN,Prior_Edu_School_Location_IJSSELSTEIN,Prior_Edu_School_Location_LEERDAM,Prior_Edu_School_Location_LEEUWARDEN,Prior_Edu_School_Location_MAASTRICHT,Prior_Edu_School_Location_NIEUWEGEIN,Prior_Edu_School_Location_NIJKERK,Prior_Edu_School_Location_NIJMEGEN,Prior_Edu_School_Location_ONBEKEND,Prior_Edu_School_Location_Others,Prior_Edu_School_Location_PAPENDRECHT,Prior_Edu_School_Location_PURMEREND,Prior_Edu_School_Location_ROTTERDAM,Prior_Edu_School_Location_SCHOONHOVEN,Prior_Edu_School_Location_SLEEUWIJK,Prior_Edu_School_Location_TERNEUZEN,Prior_Edu_School_Location_TIEL,Prior_Edu_School_Location_TILBURG,Prior_Edu_School_Location_UITHOORN,Prior_Edu_School_Location_UTRECHT,Prior_Edu_School_Location_VEENENDAAL,Prior_Edu_School_Location_VLEUTEN,Prior_Edu_School_Location_VOORBURG,Prior_Edu_School_Location_VUGHT,Prior_Edu_School_Location_WAGENINGEN,Prior_Edu_School_Location_WARNSVELD,Prior_Edu_School_Location_WIJK_BIJ_DUURSTEDE,Prior_Edu_School_Location_ZALTBOMMEL,Prior_Edu_School_Location_ZEIST,Prior_Edu_School_Location_ZOETERMEER,Prior_Edu_School_Location_ZUTPHEN,Prior_Edu_School_Location_ZWOLLE,Prior_Edu_Postcode_1033_SB,Prior_Edu_Postcode_1071

We see 246 columns left out of 301

In [23]:
data_final.to_csv(os.path.join("..", "data", "processed", "removed_high_corr.csv"), header=True)

In [24]:
data = data_final
data

,Study_Prog_Exam_Completed,Study_Prog_Group_Size,Student_Enrollment_Gap,Exit_Status,Study_Prog_Name_Accountancy,Study_Prog_Name_Allied_Medical_Care,Study_Prog_Name_Arts_Therapies,Study_Prog_Name_Bachelor_of_Law,Study_Prog_Name_Biology_Medical_Laboratory_Research,Study_Prog_Name_Built_Environment,Study_Prog_Name_Business_Administration,Study_Prog_Name_Cesar_Kinetics_Therapy,Study_Prog_Name_Chemical_Engineering,Study_Prog_Name_Chemistry,Study_Prog_Name_Communication,Study_Prog_Name_Communication_and_Multimedia_Design,Study_Prog_Name_Creative_Business,Study_Prog_Name_Dental_Hygiene,Study_Prog_Name_Educational_Theory,Study_Prog_Name_Electrical_and_Electronic_Engineering,Study_Prog_Name_Entrepreneurship_Retail_Management,Study_Prog_Name_Facility_Management,Study_Prog_Name_Finance_Control,Study_Prog_Name_Finance_Tax_and_Advice,Study_Prog_Name_Health_Care_Management,Study_Prog_Name_Human_Resource_Management,Study_Prog_Name_Industrial_Engineering_Management,Study_Prog_Name_Information_Communication_Technology,Study_Prog_Name_International_Business_Utrecht,Study_Prog_Name_Interpreter,Study_Prog_Name_Journalism,Study_Prog_Name_Logistics_Management,Study_Prog_Name_Marketing_Management,Study_Prog_Name_Mechanical_Engineering,Study_Prog_Name_Nursing,Study_Prog_Name_Optometry,Study_Prog_Name_Orthoptics,Study_Prog_Name_Pharmaceutical_Business_Administration,Study_Prog_Name_Physiotherapy,Study_Prog_Name_Primary_Education,Study_Prog_Name_Safety_and_Security_Management,Study_Prog_Name_Skin_Therapy,Study_Prog_Name_Social_Legal_Services,Study_Prog_Name_Social_Work,Study_Prog_Name_Speech_and_Language_Therapy,Study_Prog_Name_Teacher_Education,Prior_Edu_Type_BUITENL_SL,Prior_Edu_Type_EDUCATIE,Prior_Edu_Type_HAVO,Prior_Edu_Type_HBO,Prior_Edu_Type_MBO,Prior_Edu_Type_OVERIG,Prior_Edu_Type_Others,Prior_Edu_Type_VWO,Prior_Edu_School_Location_'S-GRAVENHAGE,Prior_Edu_School_Location_ALMELO,Prior_Edu_School_Location_ALMERE,Prior_Edu_School_Location_ALPHEN_AAN_DEN_RIJN,Prior_Edu_School_Location_AMERSFOORT,Prior_Edu_School_Location_AMSTELVEEN,Prior_Edu_School_Location_APELDOORN,Prior_Edu_School_Location_ASSEN,Prior_Edu_School_Location_BAARN,Prior_Edu_School_Location_BARNEVELD,Prior_Edu_School_Location_BERGEN_NH,Prior_Edu_School_Location_BREDA,Prior_Edu_School_Location_BUSSUM,Prior_Edu_School_Location_CAPELLE_AAN_DEN_IJSSEL,Prior_Edu_School_Location_DELFT,Prior_Edu_School_Location_EDE,Prior_Edu_School_Location_ENSCHEDE,Prior_Edu_School_Location_ERMELO,Prior_Edu_School_Location_ETTEN-LEUR,Prior_Edu_School_Location_GORINCHEM,Prior_Edu_School_Location_GOUDA,Prior_Edu_School_Location_GRONINGEN,Prior_Edu_School_Location_HAARLEM,Prior_Edu_School_Location_HELMOND,Prior_Edu_School_Location_HENGELO_OV,Prior_Edu_School_Location_HILVERSUM,Prior_Edu_School_Location_HOOFDDORP,Prior_Edu_School_Location_HOORN_NH,Prior_Edu_School_Location_HOUTEN,Prior_Edu_School_Location_IJSSELSTEIN,Prior_Edu_School_Location_LEERDAM,Prior_Edu_School_Location_LEEUWARDEN,Prior_Edu_School_Location_MAASTRICHT,Prior_Edu_School_Location_NIEUWEGEIN,Prior_Edu_School_Location_NIJKERK,Prior_Edu_School_Location_NIJMEGEN,Prior_Edu_School_Location_ONBEKEND,Prior_Edu_School_Location_Others,Prior_Edu_School_Location_PAPENDRECHT,Prior_Edu_School_Location_PURMEREND,Prior_Edu_School_Location_ROTTERDAM,Prior_Edu_School_Location_SCHOONHOVEN,Prior_Edu_School_Location_SLEEUWIJK,Prior_Edu_School_Location_TERNEUZEN,Prior_Edu_School_Location_TIEL,Prior_Edu_School_Location_TILBURG,Prior_Edu_School_Location_UITHOORN,Prior_Edu_School_Location_UTRECHT,Prior_Edu_School_Location_VEENENDAAL,Prior_Edu_School_Location_VLEUTEN,Prior_Edu_School_Location_VOORBURG,Prior_Edu_School_Location_VUGHT,Prior_Edu_School_Location_WAGENINGEN,Prior_Edu_School_Location_WARNSVELD,Prior_Edu_School_Location_WIJK_BIJ_DUURSTEDE,Prior_Edu_School_Location_ZALTBOMMEL,Prior_Edu_School_Location_ZEIST,Prior_Edu_School_Location_ZOETERMEER,Prior_Edu_School_Location_ZUTPHEN,Prior_Edu_School_Location_ZWOLLE,Prior_Edu_Postcode_1033_SB,Prior_Edu_Postcode_1071

In [25]:
print(data['Exit_Status'].value_counts())

Exit_Status
2    43154
1     3489
0     1748
Name: count, dtype: int64
